# Endocardial Contour Extraction by Iterative Thresholding

On suit l'algo (images normalisées) :

1. **Initial Seed Point Estimation**
2. **Mean and Standard Deviation of Blood Signal Estimation**
3. **Coil-Sensitivity Compensation**
4. **Myocardial Signal Intensity Estimation**
5. **LV Segmentation and Volume Measurement**

In [ ]:
import os
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from skimage.transform import hough_circle, hough_circle_peaks
from skimage.feature import canny
from skimage.draw import circle_perimeter
from scipy.ndimage import gaussian_filter
from scipy.optimize import leastsq

%matplotlib inline

## Loading the Data

First, let's load the data for the first 5 patients.

In [ ]:
# Define the base paths
training_base_path = 'database/training/'
testing_base_path = 'database/testing/'

# List all patient folders in the training dataset
patient_folders = [folder for folder in os.listdir(training_base_path) if os.path.isdir(os.path.join(training_base_path, folder))]
patient_folders.sort()
print(f"Found {len(patient_folders)} patients in the training dataset.")

# Select the first 5 patients
selected_patients = patient_folders[:5]
print(f"Selected patients: {selected_patients}")

## Utility Functions

In [ ]:
def load_patient_data(patient_id):
    # Define the patient folder path
    patient_folder = os.path.join(training_base_path, patient_id)
    
    # Load the 4D image (Height x Width x Slices x Time)
    image_path = os.path.join(patient_folder, f'{patient_id}_4d.nii.gz')
    image_4d = nib.load(image_path).get_fdata()
    
    # Load the ground truth masks for frame 01
    gt_path = os.path.join(patient_folder, f'{patient_id}_frame01_gt.nii.gz')
    gt_image = nib.load(gt_path).get_fdata()
    
    return image_4d, gt_image

def normalize_image(img):
    img_normalized = (img - np.min(img)) / (np.max(img) - np.min(img))
    return img_normalized

def display_image(img, title='', cmap='gray'):
    plt.imshow(img, cmap=cmap)
    plt.title(title)
    plt.axis('off')
    plt.show()

## Step 1: Initial Seed Point Estimation

In [ ]:
def estimate_initial_seed(image_4d):
    # Subtract phase 1 and phase max images
    phase1 = image_4d[:, :, :, 0]
    phaseMax = image_4d[:, :, :, -1]
    subtraction = np.abs(phase1 - phaseMax)
    
    # For simplicity, we'll work with the middle slice
    mid_slice = subtraction.shape[2] // 2
    sub_image = subtraction[:, :, mid_slice]
    orig_image = image_4d[:, :, mid_slice, 0]
    
    # Normalize the image
    sub_image_norm = normalize_image(sub_image)
    
    # Edge detection
    edges = canny(sub_image_norm, sigma=2, low_threshold=0.01, high_threshold=0.1)
    
    # Perform Hough Transform to detect circles
    hough_radii = np.arange(20, 50, 2)
    hough_res = hough_circle(edges, hough_radii)
    
    # Select the most prominent circle
    accums, cx, cy, radii = hough_circle_peaks(hough_res, hough_radii, total_num_peaks=1)
    
    if len(cx) > 0:
        seed_point = (int(cy[0]), int(cx[0]))  # (row, col)
    else:
        seed_point = (sub_image.shape[0]//2, sub_image.shape[1]//2)
    
    return seed_point, sub_image_norm, edges, (cx, cy, radii), orig_image

In [ ]:
# Initialize lists to store results
original_images = []
sub_images = []
edges_list = []
seed_points = []
circles_list = []
patient_ids = []

# Process patients
for patient_id in selected_patients:
    print(f"Processing Patient: {patient_id}")
    image_4d, gt_image = load_patient_data(patient_id)
    seed_point, sub_image, edges, circles, original = estimate_initial_seed(image_4d)
    
    original_images.append(original)
    sub_images.append(sub_image)
    edges_list.append(edges)
    seed_points.append(seed_point)
    circles_list.append(circles)
    patient_ids.append(patient_id)

# Display the subtraction images
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for i, ax in enumerate(axes):
    ax.imshow(sub_images[i], cmap='gray')
    ax.set_title(f'Patient {patient_ids[i]}')
    ax.axis('off')
plt.suptitle('Normalized Subtraction Images (Phase 1 - Phase 10)')
plt.tight_layout()
plt.show()

# Display edges
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for i, ax in enumerate(axes):
    ax.imshow(edges_list[i], cmap='gray')
    ax.set_title(f'Patient {patient_ids[i]}')
    ax.axis('off')
plt.suptitle('Edges Detected')
plt.tight_layout()
plt.show()

# Display detected circles and seed points
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for i, ax in enumerate(axes):
    ax.imshow(sub_images[i], cmap='gray')
    circles = circles_list[i]
    seed_point = seed_points[i]
    for center_y, center_x, radius in zip(circles[1], circles[0], circles[2]):
        circy, circx = circle_perimeter(int(center_y), int(center_x), int(radius))
        ax.plot(circx, circy, color='red')
    ax.set_title(f'Patient {patient_ids[i]}')
    ax.axis('off')
plt.suptitle('Detected Circles')
plt.tight_layout()
plt.show()

# Display seed points
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for i, ax in enumerate(axes):
    ax.imshow(original_images[i], cmap='gray')
    seed_point = seed_points[i]
    ax.plot(seed_point[1], seed_point[0], 'ro')
    ax.set_title(f'Patient {patient_ids[i]}')
    ax.axis('off')
plt.suptitle('Seed Points')

## Step 2: Mean and Standard Deviation of Blood Signal Estimation

In [ ]:
def region_growing(img, seed_point, threshold):
    # Initialize the segmented image
    segmented = np.zeros_like(img, dtype=bool)
    segmented[seed_point] = True
    pixel_list = [seed_point]
    
    while pixel_list:
        new_pixels = []
        for pix in pixel_list:
            for neighbor in get_neighbors(pix, img.shape):
                if not segmented[neighbor]:
                    if np.abs(img[neighbor] - img[seed_point]) < threshold:
                        segmented[neighbor] = True
                        new_pixels.append(neighbor)
        pixel_list = new_pixels
    
    return segmented

def get_neighbors(pix, shape):
    neighbors = []
    for dx in [-1, 0, 1]:
        for dy in [-1, 0, 1]:
            if dx == 0 and dy == 0:
                continue
            x_new = pix[0] + dx
            y_new = pix[1] + dy
            if x_new >= 0 and x_new < shape[0] and y_new >= 0 and y_new < shape[1]:
                neighbors.append((x_new, y_new))
    return neighbors

In [ ]:
# Initialize lists to store results
segmented_list = []
gt_images = []
lv_means = []
lv_stds = []
img_norm_list = []

for idx, patient_id in enumerate(selected_patients):
    print(f"Processing Patient: {patient_id}")
    image_4d, gt_image = load_patient_data(patient_id)
    seed_point = seed_points[idx]

    # Use phase 0 and middle slice for this example
    gt_images.append(gt_image[:, :, gt_image.shape[2] // 2])
    
    img = image_4d[:, :, image_4d.shape[2] // 2, 0]
    
    # Normalize the image
    img_norm = normalize_image(img)
    img_norm_list.append(img_norm)
    
    # Perform region growing
    threshold = 0.2
    segmented = region_growing(img_norm, seed_point, threshold)
    
    # Calculate mean and std of the LV region
    lv_pixels = img_norm[segmented]
    lv_mean = lv_pixels.mean()
    lv_std = lv_pixels.std()
    print(f"LV Mean Intensity: {lv_mean:.2f}, LV Std Dev: {lv_std:.2f}")
    
    segmented_list.append(segmented)
    lv_means.append(lv_mean)
    lv_stds.append(lv_std)

# Visualize the segmented LV regions
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for i, ax in enumerate(axes):
    ax.imshow(img_norm_list[i], cmap='gray')
    ax.contour(segmented_list[i], colors='red')
    ax.set_title(f'Patient {patient_ids[i]}')
    ax.axis('off')
plt.suptitle('Segmented LV Regions')
plt.tight_layout()
plt.show()

# Visualize the ground truth masks
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for i, ax in enumerate(axes):
    ax.imshow(gt_images[i], cmap='gray')
    ax.contour(segmented_list[i], colors='red')
    ax.set_title(f'Patient {patient_ids[i]}')
    ax.axis('off')
plt.suptitle('Ground Truth Masks')
plt.tight_layout()
plt.show()

## Step 3: Coil-Sensitivity Compensation (planar surface)

In [ ]:
def plane(p, x, y):
    a, b, c = p
    return a*x + b*y + c

def error(p, x, y, z):
    return plane(p, x, y) - z

def compensate_coil_sensitivity(img, segmented):
    # Get the coordinates and intensities of the LV pixels
    y_indices, x_indices = np.nonzero(segmented)
    intensities = img[segmented]
    
    # Fit a plane to the intensities
    initial_guess = [1, 1, 1]
    params, _ = leastsq(error, initial_guess, args=(x_indices, y_indices, intensities))
    
    # Create a meshgrid for the entire image
    Y, X = np.indices(img.shape)
    fitted_plane = plane(params, X, Y)
    
    # Compensate the image
    img_compensated = img - fitted_plane + np.mean(fitted_plane)
    
    return img_compensated, fitted_plane

In [ ]:
# Initialize list to store compensated images
img_compensated_list = []

for idx, patient_id in enumerate(selected_patients):
    print(f"Processing Patient: {patient_id}")
    
    img_norm = img_norm_list[idx]
    segmented = segmented_list[idx]
    
    # Compensate for coil sensitivity
    img_compensated, fitted_plane = compensate_coil_sensitivity(img_norm, segmented)
    img_compensated_list.append(img_compensated)

# Visualize original and compensated images
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
for i in range(5):
    axes[0, i].imshow(img_norm_list[i], cmap='gray')
    axes[0, i].set_title(f'Patient {patient_ids[i]} - Normalized')
    axes[0, i].axis('off')
    
    axes[1, i].imshow(img_compensated_list[i], cmap='gray')
    axes[1, i].set_title('Compensated Image')
    axes[1, i].axis('off')

plt.suptitle('Normalization and Coil Sensitivity Compensation')
plt.tight_layout()
plt.show()

## Step 4: Myocardial Signal Intensity Estimation

In [ ]:
def estimate_myocardium_intensity(img_compensated, seed_point):
    # Initialize variables
    # Start from a threshold slightly less than the maximum intensity in the image
    threshold = img_compensated.max() - 0.1
    thresholds = []
    volumes = []
    prev_volume = 0
    discontinuity_detected = False
    
    # Decrease threshold until minimum intensity
    while threshold > img_compensated.min():
        # Perform thresholding
        segmented = img_compensated >= threshold
        volume = np.sum(segmented)
        volumes.append(volume)
        thresholds.append(threshold)
        
        # Check for discontinuity in volume growth
        if prev_volume != 0:
            growth_ratio = (volume - prev_volume) / prev_volume
            if growth_ratio > 0.1 and not discontinuity_detected:
                # Discontinuity detected
                discontinuity_threshold = threshold
                discontinuity_detected = True
                break
        prev_volume = volume
        threshold -= 0.1  # Decrease threshold with finer granularity
    
    if discontinuity_detected:
        # Estimate myocardium mean intensity
        myoc_mean = discontinuity_threshold + 0.1  # Threshold just before the discontinuity
    else:
        # No discontinuity detected
        myoc_mean = threshold + 0.1  # Use the last threshold value
        print("Warning: Discontinuity not detected in volume growth.")
    
    return myoc_mean, thresholds, volumes


In [ ]:
# Initialize lists to store results
myoc_means = []
volume_growths = []
thresholds_list = []

for idx, patient_id in enumerate(selected_patients):
    print(f"Processing Patient: {patient_id}")
    
    img_compensated = img_compensated_list[idx]
    seed_point = seed_points[idx]
    
    # Estimate myocardium intensity
    myoc_mean, thresholds, volumes = estimate_myocardium_intensity(img_compensated, seed_point)
    print(f"Estimated Myocardium Mean Intensity: {myoc_mean:.2f}")
    
    myoc_means.append(myoc_mean)
    volume_growths.append(volumes)
    thresholds_list.append(thresholds)

# Plot volume growth for each patient
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for i, ax in enumerate(axes):
    ax.plot(thresholds_list[i], volume_growths[i])
    ax.set_title(f'Patient {patient_ids[i]}')
    ax.set_xlabel('Threshold')
    ax.set_ylabel('Volume')
    ax.invert_xaxis()
plt.suptitle('Volume Growth vs. Threshold')
plt.tight_layout()
plt.show()

## Step 5: LV Segmentation and Volume Measurement

In [ ]:
def final_segmentation(img_compensated, seed_point, myoc_mean, lv_std):
    # Absolute threshold
    n = 2
    absolute_threshold = myoc_mean + n * lv_std
    
    # Final segmentation
    segmented_lv = img_compensated >= absolute_threshold
    
    # Calculate volume (number of voxels)
    volume = np.sum(segmented_lv)
    
    return segmented_lv, volume

In [ ]:
# Initialize lists to store final segmentation results
segmented_lv_list = []
volumes = []

for idx, patient_id in enumerate(selected_patients):
    print(f"Processing Patient: {patient_id}")
    
    img_compensated = img_compensated_list[idx]
    seed_point = seed_points[idx]
    lv_std = lv_stds[idx]
    myoc_mean = myoc_means[idx]
    
    # Final segmentation
    segmented_lv, volume = final_segmentation(img_compensated, seed_point, myoc_mean, lv_std)
    print(f"Calculated LV Volume (in pixels): {volume}")
    
    segmented_lv_list.append(segmented_lv)
    volumes.append(volume)

# Visualize the final segmentation
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for i, ax in enumerate(axes):
    ax.imshow(img_norm_list[i], cmap='gray')
    ax.contour(segmented_lv_list[i], colors='red')
    ax.set_title(f'Patient {patient_ids[i]}')
    ax.axis('off')
plt.suptitle('Final LV Segmentation')
plt.tight_layout()
plt.show()